# 14. Reproducible research package

Here I collect the results that the earlier notebooks (notebooks 01–13 and the futures cache) left on Drive into **one folder**, `MyDrive/btc_lead_cost_results_77months_spot/RESEARCH_PACKAGE_v1/`. I compute nothing new in this notebook; I only gather existing result files.

| File | Contents |
|---|---|
| `DEFINITIONS.json` | Event definition, **endpoint definitions (pre-onset warning, in-progress detection and composite kept separate)**, models, 2021 training and 2022 calibration, capacity constraints, transaction-cost assumptions, multiple-testing principles |
| `EXPERIMENTS.csv` | Per notebook: question, venue, training, calibration and evaluation periods, status (confirmatory, exploratory or descriptive), pre-specified primary endpoint, tests and adjustments, SHA-256 of the spec file |
| `DATA_PROVENANCE.csv` | URL and SHA-256 of every Binance monthly archive, from each notebook's download records |
| `DATA_PROVENANCE_SUMMARY.csv` | One row per download manifest, with its row count and SHA-256 |
| `DATA_SOURCE_spot_cache.json` | Source, time range and one-minute fingerprint of the spot cache |
| `RESULTS_COUNTS.csv` | All policy comparisons in one table, with **pre-onset warning, in-progress detection and composite** as separate columns |
| `RESULTS_TESTS.csv` | Every p-value in the result files, labeled confirmatory or exploratory; within-notebook Holm-adjusted values for reference only |
| `FILE_HASHES.csv` | SHA-256 of all result files, excluding price caches and ZIP files |
| `PACKAGE_MANIFEST.json`, `00_README.md` | Package file hashes, creation time, summary |

The bundle `RESEARCH_PACKAGE_v1.zip` holds only the metadata above and **no files that contain price data**; for those (for example, the peak and crash prices in the event lists) I record only hashes.

In [1]:
from pathlib import Path
import json, hashlib, zipfile, re
import numpy as np
import pandas as pd
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'
PKG = SPOT_BASE / 'RESEARCH_PACKAGE_v1'; PKG.mkdir(parents=True, exist_ok=True)
NOW = pd.Timestamp.now(tz='UTC').isoformat()


def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fd:
        for chunk in iter(lambda: fd.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def maybe(p):
    return sha(p) if Path(p).is_file() else 'missing'


REF = json.loads((SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json').read_text())
FROZEN = json.loads((SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json').read_text())
print('package folder:', PKG)

Mounted at /content/drive
package folder: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/RESEARCH_PACKAGE_v1


## 1. Definitions (DEFINITIONS.json)

In [2]:
DEF = {
    'created_utc': NOW,
    'research_question': 'Early-warning reliability under capacity constraints: can a capacity-limited alert system warn about '
                         'BTC 4% intraday crashes before the decline starts, as opposed to detecting a decline already in progress?',
    'data': {'source': 'Binance public monthly 1-minute kline archives (data.binance.vision), spot BTCUSDT (primary), USD-M futures BTCUSDT (original study, notebook 13), '
                       'spot ETH/SOL/DOGE/AVAX (notebook 10)',
             'integrity': 'each archive checked against its published .CHECKSUM (SHA-256); see DATA_PROVENANCE.csv',
             'grid': 'complete UTC minute grid; gaps of <=5 minutes forward-filled (OHLC) with zero volume; longer gaps left missing',
             'timestamps': 'spot archives use microseconds from 2025; parsed by magnitude',
             'venue_guard': 'spot cache must match spot archive values (fingerprint) and carry DATA_SOURCE.json venue=spot'},
    'event_definition': {'rule': 'close <= 96% of the maximum close of the previous 240 minutes (breach), held for 5 consecutive closes; '
                                 'the 240 minutes before the breach fully observed; a new peak after the previous event; 24 h refractory after the previous confirmation',
                         'timestamps': {'peak': 'argmax close in the previous 240 minutes', 'onset': 'first close <= 99% of the peak (1% onset)',
                                        'breach': 'first close of the 4% breach', 'confirmed': 'fifth confirming close (real-time observable)'},
                         'alt_coins': '4% scaled by the coin\'s 2021 median daily realized volatility relative to BTC (notebook 10)',
                         'evaluation_events': 'breach in [period start + 4 h, period end) so every warning window lies inside the period'},
    'endpoints': {'early_warning_pre_onset': 'matched alert 30-240 min before the breach AND before the 1% onset',
                  'in_progress_detection_post_onset': 'matched alert 30-240 min before the breach, at or after the 1% onset',
                  'composite_caught': 'pre + post; called "caught" in notebooks 1-11. Reference only: an increase in the composite must not be read as better early warning',
                  'onset_anchored_secondary': 'alert within 240 min before the 1% onset (notebook 12)',
                  'eligibility': 'pre-onset possible only if onset-to-breach gap < 240 min; post-onset only if gap >= 30 min',
                  'matching': 'each event takes the earliest unused alert in [breach-240, breach-30] min; each alert serves at most one event',
                  'reporting_rule': 'every result table reports pre-onset and post-onset separately; the composite is never the sole basis of a claim'},
    'model': {'features': {'z_vol': 'log realized volatility over 31 one-minute returns, z-scored against the previous 30 days',
                           'tsr': 'taker sell ratio = (volume - taker buy volume) / volume in the minute'},
              'classifier': 'StandardScaler + LogisticRegression(C=0.2, class_weight=balanced, lbfgs, max_iter=400, random_state=17)',
              'label': '1 if a 4% breach occurs 30-240 min after the minute',
              'training': '2021 calendar year minus the last 245 minutes (no label leakage into 2022)',
              'calibration': '2022 only, no outcome labels: the score threshold that lets the calendar scheduler use the most alerts (cap 240), '
                             'then the strictest (fewest raw candidates); bucket policies: closest to 240 alerts, strictest on ties',
              'frozen_spot_thresholds': {'TH_CAL': REF.get('TH_CAL'), 'TH_HIGH': REF.get('TH_HIGH'), 'TH_BUCKET': FROZEN.get('TH_BUCKET')}},
    'capacity': {'primary': 'calendar scheduler: at most 20 alerts per UTC month, 240 per UTC year, 240-minute cooldown',
                 'variants': {'notebook 4': 'budgets 60/120/240/480 per year; schedulers calendar, rolling30, paced, token_bucket',
                              'notebook 6': 'constant and risk-adaptive token buckets',
                              'notebook 9': 'budget-neutral risk bucket, surge reserve (16/month base + 48/yr reserve accruing 4/month)',
                              'notebook 10': 'five coins pooled 1,200/yr and 100/month vs 240/yr each'}},
    'costs': {'notebook_1_decision_cost': 'cost points = 50 x missed event + 1 x false alarm + 10 x lateness; scenarios in weight_sensitivity.csv; hypothetical weights, not money',
              'notebook_4_stylized_hedge': 'on each alert reduce exposure by 50% for 240 min; enter at the next positive-volume minute (<=5 min wait); '
                                           'overlapping hedges merged; one-way fee 10 bp primary, grid 0/5/10/25 bp on each exposure change; '
                                           'compared with an exposure-matched buy-and-hold; illustrative, not live trading',
              'break_even': 'notebook 4 calibration_break_even_precision.csv'},
    'multiple_testing': {'confirmatory': 'only notebook 11 (2018-2020 one-shot holdout): H1 one-sided circular-shift permutation (alpha 0.05); '
                                         'H2 three one-sided exact McNemar tests with Holm; H3 descriptive; H4 none selected',
                         'exploratory': 'all other p-values are unadjusted and descriptive (2023-2026 was inspected repeatedly); '
                                        'RESULTS_TESTS.csv adds a within-notebook Holm column for reference only',
                         'clustered_events': 'notebook 12 re-tests pooled vs separate budgets with shock- and month-level sign-flip permutation'},
    'status_vocabulary': {'confirmatory': 'hypothesis, thresholds and test fixed (hashed) before the data were read; evaluated once (lock file)',
                          'frozen replay': 'fixed policy replayed on later data (notebook 7); not a never-seen sample',
                          'exploratory': 'evaluated on data already inspected', 'descriptive': 'no hypothesis test',
                          'post-hoc audit': 'checks assumptions of a locked test without changing it'}}
(PKG / 'DEFINITIONS.json').write_text(json.dumps(DEF, indent=2, ensure_ascii=False, default=str))
print('DEFINITIONS.json written')

DEFINITIONS.json written


## 2. Experiment table (EXPERIMENTS.csv)

In [3]:
S, F = SPOT_BASE, FUT_BASE
DEV = '2023-01-01..2026-05-01'
EXP = [
    dict(id='F1-F7', notebook='original futures notebooks 1-7', venue='futures (USD-M)', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits (composite also reported)', tests='paired exact McNemar (unadjusted)',
         folder=str(F), spec=''),
    dict(id='1', notebook='1_BTCUSDT_77_Monthly_LeadTime_Cost_Colab_SPOT', venue='spot', train='2021-2022 (rule selection)', calibration='-',
         evaluation='2023-01-01..2025-01-01 (2025-2026 period skipped: cache ends 2026-04)', status='exploratory',
         primary_endpoint='decision cost points (miss 50 / false 1 / late 10)', tests='none', folder=str(S), spec=''),
    dict(id='2', notebook='2_BTC_Event_Definition_Eligibility_Audit_Colab_SPOT', venue='spot', train='-', calibration='-', evaluation='2020-01..2026-04',
         status='descriptive', primary_endpoint='event catalog', tests='none', folder=str(S / 'event_definition_eligibility_audit'), spec=''),
    dict(id='3', notebook='3_BTC_Gate_Ablation_PriceOnly_Colab_SPOT', venue='spot', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits', tests='paired exact McNemar (unadjusted)', folder=str(S / 'gate_ablation_price_only_v1'), spec=''),
    dict(id='4', notebook='4_BTC_Systems_Research_Colab_SPOT', venue='spot', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits at B=240, calendar scheduler', tests='paired exact tests, month-block bootstrap (unadjusted)',
         folder=str(S / 'systems_engineering_scheduler_oracle_value_v1'), spec='study_specification.json'),
    dict(id='5', notebook='5_BTC_Causal_Alert_Selection_Colab_SPOT', venue='spot', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits', tests='paired exact McNemar (unadjusted)', folder=str(S / 'causal_alert_selection_v1'),
         spec='alert_selection_spec.json'),
    dict(id='6', notebook='6_BTC_Final_Risk_Adaptive_Capacity_Colab_SPOT', venue='spot', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits', tests='paired exact McNemar (unadjusted)', folder=str(S / 'FINAL_risk_adaptive_alert_capacity_v1'),
         spec='final_adaptive_study_spec.json'),
    dict(id='7', notebook='7_BTC_2026_Validation_SourceFix_v2_SPOT', venue='spot', train='2021', calibration='2022 (thresholds frozen)',
         evaluation='2026-05-01..2026-10-01', status='frozen replay', primary_endpoint='pre-onset hits', tests='paired exact (unadjusted)',
         folder=str(S / 'FINAL_frozen_policy_May_Sep_2026_v2_source_verified'), spec='FROZEN_REPLAY_SPEC.json'),
    dict(id='8', notebook='8_BTC_CrossVenue_Information_SPOT', venue='spot + futures features', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits (candidate rule for 11)', tests='paired exact McNemar (unadjusted)',
         folder=str(S / 'crossvenue_information_v1'), spec='crossvenue_spec.json'),
    dict(id='9', notebook='9_BTC_Capacity_Design_SPOT', venue='spot', train='2021', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits (candidate rule for 11)', tests='paired exact McNemar (unadjusted)',
         folder=str(S / 'capacity_design_v1'), spec='capacity_spec.json'),
    dict(id='10', notebook='10_MultiCoin_Shared_Budget_SPOT', venue='spot BTC/ETH/SOL/DOGE/AVAX', train='2021 (per coin)', calibration='2022', evaluation=DEV,
         status='exploratory', primary_endpoint='pre-onset hits summed over coins', tests='paired exact McNemar over coin-events (unadjusted; events clustered, see 12)',
         folder=str(S / 'multicoin_shared_budget_v1'), spec='multicoin_spec.json'),
    dict(id='11', notebook='11_BTC_Holdout_2018_2020_Confirmatory_SPOT', venue='spot', train='2021 (backcast)', calibration='2022',
         evaluation='2018-01-01..2021-01-01', status='confirmatory (one-shot, locked)',
         primary_endpoint='pre-onset hits (H1); scheduler comparison (H2)', tests='H1 circular-shift permutation; H2 McNemar + Holm(3)',
         folder=str(S / 'HOLDOUT_2018_2020_confirmatory_v1'), spec='HOLDOUT_SPEC.json'),
    dict(id='12', notebook='12_BTC_EarlyWarning_Reliability_SPOT', venue='spot (+ 5 coins for pooling)', train='2021', calibration='2022',
         evaluation=f'{DEV} and 2018-01-01..2021-01-01', status='exploratory + post-hoc audit',
         primary_endpoint='pre-onset and post-onset separately', tests='circular shift (bands), regime-matched null, cluster sign-flip (unadjusted)',
         folder=str(S / 'early_warning_reliability_v1'), spec='RELIABILITY_SPEC.json'),
    dict(id='13', notebook='13_BTC_Spot_vs_Futures_Same_Events', venue='spot and futures', train='2021', calibration='2022', evaluation=DEV,
         status='descriptive', primary_endpoint='order of price milestones and first signals in matched events', tests='none',
         folder=str(S / 'spot_futures_same_event_v1'), spec='SAME_EVENT_SPEC.json'),
]
EX = pd.DataFrame(EXP)
EX['spec_sha256'] = [maybe(Path(r.folder) / r.spec) if r.spec else '' for r in EX.itertuples()]
EX['folder_exists'] = [Path(r.folder).is_dir() for r in EX.itertuples()]
EX.to_csv(PKG / 'EXPERIMENTS.csv', index=False); display(EX[['id', 'status', 'evaluation', 'primary_endpoint', 'spec_sha256', 'folder_exists']])

,id,status,evaluation,primary_endpoint,spec_sha256,folder_exists
0,F1-F7,exploratory,2023-01-01..2026-05-01,pre-onset hits (composite also reported),,True
1,1,exploratory,2023-01-01..2025-01-01 (2025-2026 period skipp...,decision cost points (miss 50 / false 1 / late...,,True
2,2,descriptive,2020-01..2026-04,event catalog,,True
3,3,exploratory,2023-01-01..2026-05-01,pre-onset hits,,True
4,4,exploratory,2023-01-01..2026-05-01,"pre-onset hits at B=240, calendar scheduler",f5d82820355fe19988c81b2c5f3b69f454986e69403149...,True
5,5,exploratory,2023-01-01..2026-05-01,pre-onset hits,d3978e1530cb4801e0b0172f6ecf0ea02cb8d49fb9c828...,True
6,6,exploratory,2023-01-01..2026-05-01,pre-onset hits,1703156f64803c870af65fcc7a0b1d12d1bdd6bef1e6b7...,True
7,7,frozen replay,2026-05-01..2026-10-01,pre-onset hits,3b3c50393b028d5b98d4103bea51170e56f73c042bd5a6...,True
8,8,exploratory,2023-01-01..2026-05-01,pre-onset hits (candidate rule for 11),a3561a711d906ef2b09d5a86cea218b5f9fb17e7709e96...,True
9,9,exploratory,2023-01-01..2026-05-01,pre-onset hits (candidate rule for 11),a7f0ea2bcd71d0d65c677fd042d625e73cb4cf093394de...,True


## 3. Data provenance (DATA_PROVENANCE.csv)

In [4]:
MAN = [('spot 2020-01..2026-04 (notebook 1)', S / 'spot_download_manifest.csv'), ('spot cache monthly files (notebook 1)', S / 'monthly_file_manifest.csv'),
       ('futures cache monthly files (original notebook 1)', F / 'monthly_file_manifest.csv'),
       ('spot 2017-08..2020-12 holdout (notebook 11)', S / 'HOLDOUT_2018_2020_confirmatory_v1' / 'spot_download_manifest.csv'),
       ('futures 2019-09..2020-12 holdout (notebook 11, only if used)', S / 'HOLDOUT_2018_2020_confirmatory_v1' / 'futures_download_manifest.csv'),
       ('2026 replay archives (notebook 7)', S / 'FINAL_frozen_policy_May_Sep_2026_v2_source_verified' / 'source_archive_manifest.csv')]
MAN += [(f'{c} spot (notebook 10)', S / 'multicoin_shared_budget_v1' / f'{c}_download_manifest.csv') for c in ['ETHUSDT', 'SOLUSDT', 'DOGEUSDT', 'AVAXUSDT']]
parts, prov_sum = [], []
for label, p in MAN:
    if p.is_file():
        d = pd.read_csv(p); d.insert(0, 'manifest', label); d.insert(1, 'manifest_file', str(p.relative_to(DRIVE))); parts.append(d)
        prov_sum.append(dict(manifest=label, rows=len(d), file_sha256=sha(p), has_sha256_column=any('sha' in c.lower() for c in d.columns)))
    else:
        prov_sum.append(dict(manifest=label, rows=0, file_sha256='missing', has_sha256_column=False))
PROV = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
PROV.to_csv(PKG / 'DATA_PROVENANCE.csv', index=False)
PS = pd.DataFrame(prov_sum); PS.to_csv(PKG / 'DATA_PROVENANCE_SUMMARY.csv', index=False); display(PS)
src = S / 'cache' / 'DATA_SOURCE.json'
if src.is_file():
    (PKG / 'DATA_SOURCE_spot_cache.json').write_text(src.read_text())

,manifest,rows,file_sha256,has_sha256_column
0,spot 2020-01..2026-04 (notebook 1),76,2a4f3062f5fb4613e64f10291f5dbf3bd9a1d109327881...,True
1,spot cache monthly files (notebook 1),76,ff52d71deba9443ff52aae6ef1c8733c7bdd357e2e0d0b...,False
2,futures cache monthly files (original notebook 1),76,eb3ee34c20693916ef17fbdff82e2846b86f81f01808e5...,False
3,spot 2017-08..2020-12 holdout (notebook 11),41,42218edfda5ae15995d1a959608ad7cd7256b22e677414...,True
4,"futures 2019-09..2020-12 holdout (notebook 11,...",0,missing,False
5,2026 replay archives (notebook 7),6,17b922e02f92c893c89646734a1f964b68bba5e14cf084...,True
6,ETHUSDT spot (notebook 10),76,2403640af65e62b8a6d88f0ce014ae34d97a263bce9fec...,True
7,SOLUSDT spot (notebook 10),76,8c7315d5b89bc86895215f788354d6aef9e28a65cec607...,True
8,DOGEUSDT spot (notebook 10),76,275b79fbe19a72e6bbdffcd752b63a4b3b8d110bed9008...,True
9,AVAXUSDT spot (notebook 10),76,03d7e296ab99503194a8a3d38e09129c5264c9bd7173b2...,True


## 4. Result counts (RESULTS_COUNTS.csv), pre-onset warning and in-progress detection separately

In [5]:
PRE_COLS = ['before_1pct', 'before_1pct_onset', 'pre_onset_caught', 'caught_pre1pct', 'pre_onset', 'pre_hits', 'pre1pct']
CAUGHT_COLS = ['caught', 'caught_any', 'caught_composite']
ALARM_COLS = ['alarms', 'alerts_may_sep']
LABEL_COLS = ['policy', 'mode', 'model', 'scheduler', 'name', 'arm', 'B', 'coin', 'year', 'period', 'venue', 'one_way_fee_bps']
COUNT_SRC = [
    ('F3', 'futures', DEV, 'exploratory', F / 'gate_ablation_price_only_v1' / 'gate_ablation_policy_comparison.csv'),
    ('F4', 'futures', DEV, 'exploratory', F / 'systems_engineering_scheduler_oracle_value_v1' / 'scheduler_policy_comparison.csv'),
    ('F5', 'futures', DEV, 'exploratory', F / 'causal_alert_selection_v1' / 'selection_policy_comparison.csv'),
    ('F6', 'futures', DEV, 'exploratory', F / 'FINAL_risk_adaptive_alert_capacity_v1' / 'risk_adaptive_policy_comparison.csv'),
    ('F7', 'futures', '2026-05..2026-09', 'frozen replay', F / 'FINAL_frozen_policy_May_Sep_2026_v2_source_verified' / 'FROZEN_policy_comparison_May_Sep_2026.csv'),
    ('3', 'spot', DEV, 'exploratory', S / 'gate_ablation_price_only_v1' / 'gate_ablation_policy_comparison.csv'),
    ('4', 'spot', DEV, 'exploratory', S / 'systems_engineering_scheduler_oracle_value_v1' / 'scheduler_policy_comparison.csv'),
    ('5', 'spot', DEV, 'exploratory', S / 'causal_alert_selection_v1' / 'selection_policy_comparison.csv'),
    ('6', 'spot', DEV, 'exploratory', S / 'FINAL_risk_adaptive_alert_capacity_v1' / 'risk_adaptive_policy_comparison.csv'),
    ('7', 'spot', '2026-05..2026-09', 'frozen replay', S / 'FINAL_frozen_policy_May_Sep_2026_v2_source_verified' / 'FROZEN_policy_comparison_May_Sep_2026.csv'),
    ('8', 'spot', DEV, 'exploratory', S / 'crossvenue_information_v1' / 'crossvenue_model_comparison.csv'),
    ('9', 'spot', DEV, 'exploratory', S / 'capacity_design_v1' / 'capacity_policy_comparison.csv'),
    ('10', 'spot 5 coins', DEV, 'exploratory', S / 'multicoin_shared_budget_v1' / 'multicoin_policy_by_coin.csv'),
    ('11', 'spot', '2018-01..2020-12', 'confirmatory (secondary rows)', S / 'HOLDOUT_2018_2020_confirmatory_v1' / 'secondary_policy_results.csv'),
    ('12', 'spot', f'{DEV} and 2018-01..2020-12', 'exploratory', S / 'early_warning_reliability_v1' / '1_endpoint_table.csv'),
    ('13', 'spot+futures', DEV, 'descriptive', S / 'spot_futures_same_event_v1' / '1_reproduced_baselines.csv'),
]
rows, missing = [], []
for exp, venue, period, status, p in COUNT_SRC:
    if not p.is_file():
        missing.append(dict(experiment=exp, file=str(p), issue='missing')); continue
    d = pd.read_csv(p)
    pc = next((c for c in PRE_COLS if c in d.columns), None); cc = next((c for c in CAUGHT_COLS if c in d.columns), None)
    ac = next((c for c in ALARM_COLS if c in d.columns), None)
    if pc is None or cc is None or ac is None:
        missing.append(dict(experiment=exp, file=str(p), issue=f'schema not recognised: {list(d.columns)[:12]}')); continue
    labs = [c for c in LABEL_COLS if c in d.columns]
    for rr in d.to_dict('records'):
        label = ' | '.join(f'{c}={rr[c]}' for c in labs if pd.notna(rr[c]))
        pre = float(rr[pc]) if pd.notna(rr[pc]) else np.nan; cau = float(rr[cc]) if pd.notna(rr[cc]) else np.nan
        rows.append(dict(experiment=exp, venue=venue, period=period, status=status, row=label, alarms=rr[ac],
                         early_warning_pre_onset=pre, in_progress_detection_post_onset=cau - pre, composite_caught_reference=cau,
                         events=rr.get('events', rr.get('new_crashes', np.nan)), source=str(p.relative_to(DRIVE))))
RC = pd.DataFrame(rows); RC.to_csv(PKG / 'RESULTS_COUNTS.csv', index=False)
print(f'RESULTS_COUNTS: {len(RC)} rows from {RC.source.nunique() if len(RC) else 0} files')
if missing:
    display(pd.DataFrame(missing))

RESULTS_COUNTS: 226 rows from 16 files


## 5. Tests (RESULTS_TESTS.csv)

In [6]:
# (experiment, status, file, endpoint when neither the row nor the p-column name says it)
# notebooks 3/4/5 "exact" p-values are computed on caught-event sets (composite); notebook 11 H2 is on pre-onset hits.
TEST_SRC = [
    ('F3', 'exploratory', F / 'gate_ablation_price_only_v1' / 'paired_event_gate_and_spectral.csv', 'composite'),
    ('F4', 'exploratory', F / 'systems_engineering_scheduler_oracle_value_v1' / 'paired_scheduler_comparison.csv', 'composite'),
    ('F5', 'exploratory', F / 'causal_alert_selection_v1' / 'paired_selection_results.csv', 'composite'),
    ('F6', 'exploratory', F / 'FINAL_risk_adaptive_alert_capacity_v1' / 'paired_risk_adaptive_comparison.csv', None),
    ('F7', 'frozen replay', F / 'FINAL_frozen_policy_May_Sep_2026_v2_source_verified' / 'FROZEN_paired_exact_results.csv', None),
    ('3', 'exploratory', S / 'gate_ablation_price_only_v1' / 'paired_event_gate_and_spectral.csv', 'composite'),
    ('4', 'exploratory', S / 'systems_engineering_scheduler_oracle_value_v1' / 'paired_scheduler_comparison.csv', 'composite'),
    ('5', 'exploratory', S / 'causal_alert_selection_v1' / 'paired_selection_results.csv', 'composite'),
    ('6', 'exploratory', S / 'FINAL_risk_adaptive_alert_capacity_v1' / 'paired_risk_adaptive_comparison.csv', None),
    ('7', 'frozen replay', S / 'FINAL_frozen_policy_May_Sep_2026_v2_source_verified' / 'FROZEN_paired_exact_results.csv', None),
    ('8', 'exploratory', S / 'crossvenue_information_v1' / 'paired_vs_M0.csv', None),
    ('9', 'exploratory', S / 'capacity_design_v1' / 'paired_vs_P0.csv', None),
    ('10', 'exploratory', S / 'multicoin_shared_budget_v1' / 'paired_pooled_vs_separate.csv', None),
    ('11', 'confirmatory', S / 'HOLDOUT_2018_2020_confirmatory_v1' / 'H2_scheduler_tests.csv', 'pre'),
    ('12', 'exploratory / post-hoc audit', S / 'early_warning_reliability_v1' / '2_endpoint_nulls.csv', None),
    ('12', 'post-hoc audit', S / 'early_warning_reliability_v1' / '3bc_null_audit.csv', None),
    ('12', 'exploratory', S / 'early_warning_reliability_v1' / '4_pooling_by_endpoint_and_cluster.csv', None),
]
CLASS = {'pre': 'early warning (pre-onset)', 'post': 'in-progress detection (post-onset)', 'composite': 'composite (reference)',
         'onset_anchored': 'early warning (onset-anchored, secondary)', 'episodes_pre': 'early warning (pre-onset, episode level)',
         'episodes_caught': 'composite (reference, episode level)'}
ROW_VALUE = {'pre': 'pre', 'early_hit': 'pre', 'post': 'post', 'caught': 'composite', 'hit': 'composite',
             'onset_anchored': 'onset_anchored', 'episodes_pre': 'episodes_pre', 'episodes_caught': 'episodes_caught'}


def endpoint_class(col, row, default):
    for key in ('endpoint', 'metric'):
        if key in row and str(row[key]) in ROW_VALUE:
            return CLASS[ROW_VALUE[str(row[key])]]
    c = col.lower()
    if re.search(r'pre1pct|preonset|pre_onset', c):
        return CLASS['pre']
    if re.search(r'caught|any_hit', c):
        return CLASS['composite']
    return CLASS[default] if default else 'unspecified (see source file)'


trows = []
for exp, status, p, default in TEST_SRC:
    if not p.is_file():
        trows.append(dict(experiment=exp, status=status, comparison='', statistic='', p_value=np.nan, correction='', endpoint_class='', source=str(p), note='missing'))
        continue
    d = pd.read_csv(p)
    pcols = [c for c in d.columns if (c.startswith('p_') or c.endswith('_p') or 'pvalue' in c.lower() or c == 'p') and pd.api.types.is_numeric_dtype(d[c])]
    labs = [c for c in d.columns if c not in pcols and not pd.api.types.is_float_dtype(d[c])][:6]
    for rr in d.to_dict('records'):
        label = ' | '.join(f'{c}={rr[c]}' for c in labs)
        for c in pcols:
            trows.append(dict(experiment=exp, status=status, comparison=label, statistic=c, p_value=rr[c],
                              correction='Holm (in file)' if 'holm' in c.lower() else 'none',
                              endpoint_class=endpoint_class(c, rr, default), source=str(p.relative_to(DRIVE)), note=''))
hj = S / 'HOLDOUT_2018_2020_confirmatory_v1' / 'HOLDOUT_RESULTS.json'
if hj.is_file():
    H1 = json.loads(hj.read_text())['H1']
    trows.append(dict(experiment='11', status='confirmatory', comparison='H1 calendar vs circular-shift null', statistic='p_one_sided (pre-onset hits)',
                      p_value=H1['p_one_sided'], correction='none (single primary)', endpoint_class='early warning (pre-onset)', source=str(hj.relative_to(DRIVE)), note=''))
    trows.append(dict(experiment='11', status='confirmatory (secondary)', comparison='H1 calendar vs circular-shift null', statistic='p_any_hit_secondary',
                      p_value=H1['p_any_hit_secondary'], correction='none', endpoint_class='composite (reference)', source=str(hj.relative_to(DRIVE)), note=''))
RT = pd.DataFrame(trows)


def holm(ps):
    ps = np.asarray(ps, float); order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * ps[i])); adj[i] = run
    return adj


RT['holm_within_notebook_reference'] = np.nan
for exp, g in RT[RT.p_value.notna() & ~RT.status.str.startswith('confirmatory') & (RT.correction == 'none')].groupby('experiment'):
    RT.loc[g.index, 'holm_within_notebook_reference'] = holm(g.p_value.to_numpy())
RT.to_csv(PKG / 'RESULTS_TESTS.csv', index=False)
print(f'RESULTS_TESTS: {int(RT.p_value.notna().sum())} p-values'); display(RT.groupby(['experiment', 'endpoint_class']).size().rename('p-values').reset_index())

RESULTS_TESTS: 314 p-values


,experiment,endpoint_class,p-values
0,10,composite (reference),2
1,10,early warning (pre-onset),2
2,11,composite (reference),1
3,11,early warning (pre-onset),7
4,12,composite (reference),19
5,12,"composite (reference, episode level)",16
6,12,"early warning (onset-anchored, secondary)",16
7,12,early warning (pre-onset),19
8,12,"early warning (pre-onset, episode level)",16
9,12,in-progress detection (post-onset),19


## 6. File hashes (FILE_HASHES.csv), excluding price caches and ZIP files

In [7]:
PRICE_HINT = re.compile(r'(^|_)(open|high|low|close|price|peak_close|breach_close|vwap)($|_)', re.I)
frows = []
for base in [S, F]:
    for p in sorted(base.rglob('*')):
        if not p.is_file() or 'cache' in p.relative_to(base).parts or p.suffix.lower() in ('.zip', '.parquet') or PKG in p.parents:
            continue
        price = False
        if p.suffix.lower() == '.csv':
            try:
                cols = pd.read_csv(p, nrows=0).columns
                price = any(PRICE_HINT.search(c) for c in cols)
            except Exception:
                pass
        frows.append(dict(path=str(p.relative_to(DRIVE)), bytes=p.stat().st_size, sha256=sha(p), contains_price_columns=price))
FH = pd.DataFrame(frows); FH.to_csv(PKG / 'FILE_HASHES.csv', index=False)
print(f'{len(FH)} result files hashed; {int(FH.contains_price_columns.sum())} contain price columns (hash only, not in the zip)')

322 result files hashed; 13 contain price columns (hash only, not in the zip)


## 7. Summary README, manifest and ZIP

In [8]:
def j(p):
    return json.loads(p.read_text()) if p.is_file() else None


rel = S / 'early_warning_reliability_v1'
lines = ['# BTC crash early-warning research package (v1)', '', f'Created {NOW}.', '',
         '**Research question.** ' + DEF['research_question'], '',
         '**Reporting rule.** Early warning (alert before the 1% onset) and in-progress detection (alert after the onset but at least 30 min before the 4% breach) '
         'are reported separately. The composite "caught" is reference only.', '']
hres = j(S / 'HOLDOUT_2018_2020_confirmatory_v1' / 'HOLDOUT_RESULTS.json')
if hres:
    h = hres['H1']
    lines += ['## Confirmatory holdout (2018-2020, notebook 11)', '',
              f"- H1 early warning: {h['pre_onset_hits']} pre-onset hits vs null mean {h['null_mean_pre_onset']:.1f}, one-sided p = {h['p_one_sided']:.4f}",
              f"- Composite (secondary): {h['any_hits']} caught, p = {h['p_any_hit_secondary']:.4g}",
              f"- H2 scheduler: " + '; '.join(f"{r['comparison']} Holm p = {r['p_holm']:.3f}" for r in hres['H2']),
              f"- H3 capacity: {hres['H3'].get('missed_no_room')} of {hres['H3'].get('missed_no_room', 0) + hres['H3'].get('missed_some_room', 0)} misses had no monthly room", '']
if (rel / '2_endpoint_nulls.csv').is_file():
    t2 = pd.read_csv(rel / '2_endpoint_nulls.csv')
    lines += ['## Endpoints separated (notebook 12, primary circular-shift null)', '', '| period | endpoint | observed | null mean | one-sided p |', '|---|---|---|---|---|']
    for r in t2[t2.endpoint.isin(['pre', 'post', 'onset_anchored'])].itertuples():
        lines.append(f'| {r.period} | {r.endpoint_class} | {r.observed} | {r.null_mean:.1f} | {r.p_one_sided:.4f} |')
    lines.append('')
lines += ['## Files', ''] + [f'- `{n}`' for n in ['DEFINITIONS.json', 'EXPERIMENTS.csv', 'DATA_PROVENANCE.csv', 'DATA_PROVENANCE_SUMMARY.csv',
                                                  'RESULTS_COUNTS.csv', 'RESULTS_TESTS.csv', 'FILE_HASHES.csv', 'PACKAGE_MANIFEST.json']]
(PKG / '00_README.md').write_text('\n'.join(lines) + '\n')

members = [p for p in sorted(PKG.iterdir()) if p.is_file() and p.suffix != '.zip' and p.name != 'PACKAGE_MANIFEST.json']
manifest = dict(created_utc=NOW, files={p.name: sha(p) for p in members},
                experiments_spec_sha256=dict(zip(EX.id, EX.spec_sha256)), result_files_hashed=len(FH),
                note='price data (caches, archives, files with price columns) are hashed in FILE_HASHES.csv but not included')
(PKG / 'PACKAGE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
zp = PKG / 'RESEARCH_PACKAGE_v1.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in members + [PKG / 'PACKAGE_MANIFEST.json']:
        z.write(p, arcname=f'RESEARCH_PACKAGE_v1/{p.name}')
print('package written:', zp, f'({zp.stat().st_size / 1024:.0f} KB)')
print('\n'.join(lines[:30]))

package written: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/RESEARCH_PACKAGE_v1/RESEARCH_PACKAGE_v1.zip (57 KB)
# BTC crash early-warning research package (v1)

Created 2026-10-29T14:19:40.052699+00:00.

**Research question.** Early-warning reliability under capacity constraints: can a capacity-limited alert system warn about BTC 4% intraday crashes before the decline starts, as opposed to detecting a decline already in progress?

**Reporting rule.** Early warning (alert before the 1% onset) and in-progress detection (alert after the onset but at least 30 min before the 4% breach) are reported separately. The composite "caught" is reference only.

## Confirmatory holdout (2018-2020, notebook 11)

- H1 early warning: 13 pre-onset hits vs null mean 11.4, one-sided p = 0.3532
- Composite (secondary): 36 caught, p = 9.999e-05
- H2 scheduler: calendar_vs_rolling30 Holm p = 1.000; calendar_vs_paced Holm p = 1.000; calendar_vs_token_bucket Holm p = 0.516
- H3 capacity: 103 of 